In [2]:
import requests

class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)

        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")
        
        return response.json()['choices'][0]['message']['content'].strip()


In [3]:
import random
import json
import csv
from typing import List, Optional

class PromptBuilder:
    def __init__(self, train_data_path: str, schema_csv_path: Optional[str] = None, mode: str = 'zero-shot', shot_num: int = 0):
        self.mode = mode
        self.shot_num = shot_num

        with open(train_data_path, 'r', encoding='utf-8') as f:
            self.examples = [json.loads(line.strip()) for line in f]

        self.schema_text = self._read_schema(schema_csv_path) if schema_csv_path else ""

    def _read_schema(self, schema_csv_path: str) -> str:
        schema_map = {}
        with open(schema_csv_path, newline='', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                table = row['Table Name'].strip()
                field = row[' Field Name'].strip()
                if table == '-' or field == '-':
                    continue
                if table not in schema_map:
                    schema_map[table] = []
                schema_map[table].append(field)
        return "\n".join([f"{table}({', '.join(fields)})" for table, fields in schema_map.items()])

    def build_prompt(self, input_question: str) -> str:
        header = (
            "You are a SQL query generator.\n"
            "Given a question and a database schema, translate the question into a valid SQL query.\n"
            "Only output the SQL code.\n\n"
            f"Schema:\n{self.schema_text}\n\n"
        )
        fewshot_section = ""
        if self.mode != 'zero-shot':
            samples = random.sample(self.examples, min(self.shot_num, len(self.examples)))
            for ex in samples:
                fewshot_section += f"Question: {ex['text']}\nSQL: {ex['sql']}\n\n"

        final_question = f"Question: {input_question}\nSQL:"
        return header + fewshot_section + final_question
    # def build_prompt(self, input_question: str) -> str:
    #     header = (
    #         "You are a SQL query generator.\n"
    #         "Translate the given natural language question into a SQL query.\n"
    #         "The SQL must match the database schema below.\n"
    #         "Do not explain. Only output valid SQL code.\n\n"
    #         f"Schema:\n{self.schema_text}\n\n"
    #     )

    #     instruction = (
    #         "-- Examples --\n" if self.mode != 'zero-shot' else ""
    #     )

    #     fewshot_section = ""
    #     if self.mode != 'zero-shot':
    #         samples = random.sample(self.examples, min(self.shot_num, len(self.examples)))
    #         for ex in samples:
    #             fewshot_section += f"# Question: {ex['text']}\nSELECT {ex['sql']}\n\n"

    #     final_question = f"# Question: {input_question}\nSELECT"
    #     return header + instruction + fewshot_section + final_question


# Example usage
if __name__ == '__main__':
    builder = PromptBuilder(train_data_path="generation_train.jsonl", schema_csv_path="atis-schema.csv",
    mode="zero-shot",shot_num=5)
    q = "what are the flights from dallas to boston on july 5"
    prompt = builder.build_prompt(q)
    print(prompt)

You are a SQL query generator.
Given a question and a database schema, translate the question into a valid SQL query.
Only output the SQL code.

Schema:
AIRCRAFT(AIRCRAFT_CODE, AIRCRAFT_DESCRIPTION, MANUFACTURER, BASIC_TYPE, ENGINES, PROPULSION, WIDE_BODY, WING_SPAN, LENGTH, WEIGHT, CAPACITY, PAY_LOAD, CRUISING_SPEED, RANGE_MILES, PRESSURIZED)
AIRLINE(AIRLINE_CODE, AIRLINE_NAME, NOTE)
AIRPORT(AIRPORT_CODE, AIRPORT_NAME, AIRPORT_LOCATION, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE, MINIMUM_CONNECT_TIME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE, MILES_DISTANT, DIRECTION, MINUTES_DISTANT)
CITY(CITY_CODE, CITY_NAME, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE)
CLASS_OF_SERVICE(BOOKING_CLASS, RANK, CLASS_DESCRIPTION)
CODE_DESCRIPTION(CODE, DESCRIPTION)
COMPARTMENT_CLASS(COMPARTMENT, CLASS_TYPE)
DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)
DAYS(DAYS_CODE, DAY_NAME)
DUAL_CARRIER(MAIN_AIRLINE, LOW_FLIGHT_NUMBER, HIGH_FLIGHT_NUMBER, DUAL_AIRLINE, SERVICE_NAME)
EQUIPMENT_SEQUENCE(AIRCRAFT_CODE_S

In [4]:
import json
import re
from tqdm import tqdm
# from promptbuilder_with_schema import PromptBuilder
# from openrouter_sdk import OpenRouter

API_KEY = "YOUR_OPENROUTER_API_KEY"
MODEL = "meta-llama/llama-3.2-3b-instruct"
client = OpenRouter(api_key=API_KEY, model=MODEL)

def clean_sql(sql):
    return re.sub(r'\s+', ' ', sql.strip().lower())

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line.strip()) for line in f if line.strip()]

def evaluate(pred, refs):
    pred = clean_sql(pred)
    refs = [clean_sql(r) for r in refs]
    return pred in refs

def run_debug(train_file, test_file, schema_csv_file, shot_type="few-shot", shot_count=5, n=5):
    builder = PromptBuilder(train_file, schema_csv_file, mode=shot_type, shot_num=shot_count)
    test_data = load_jsonl(test_file)[:n]

    correct = 0
    log_lines = []

    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item['text'])
        print(f"\n--------------------------- Prompt #{idx+1}")
        print(prompt)
        print(f"\n[Token Length]: {len(prompt.split())} words")

        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue
        gen_sql+=" ;"
        gold = item['sql']
        hit = evaluate(gen_sql, gold)
        correct += int(hit)

        print(f"🧪 Q: {item['text']}")
        print(f"🔧 SQL_PRED: {gen_sql}")
        print(f"🎯 SQL_GOLD: {gold}")
        print(f"✅ MATCH: {hit}")

        log_lines.append(json.dumps({
            "question": item['text'],
            "prompt": prompt,
            "pred": gen_sql,
            "gold": gold,
            "match": hit
        }))

    acc = correct / len(test_data)
    print(f"\n✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{len(test_data)})")

    with open("prompt_debug_log.jsonl", "w", encoding="utf-8") as f:
        for line in log_lines:
            f.write(line + "\n")

    print("\n📝 Log saved to prompt_debug_log.jsonl")


In [5]:
# from debug_prompt_runner import run_debug

run_debug(
    train_file="generation_train.jsonl",
    test_file="generation_test.jsonl",
    schema_csv_file="atis-schema.csv",
    shot_type="many-shot",
    shot_count=5,
    n=3  # 控制测试样本数
)


  0%|          | 0/3 [00:00<?, ?it/s]


--------------------------- Prompt #1
You are a SQL query generator.
Given a question and a database schema, translate the question into a valid SQL query.
Only output the SQL code.

Schema:
AIRCRAFT(AIRCRAFT_CODE, AIRCRAFT_DESCRIPTION, MANUFACTURER, BASIC_TYPE, ENGINES, PROPULSION, WIDE_BODY, WING_SPAN, LENGTH, WEIGHT, CAPACITY, PAY_LOAD, CRUISING_SPEED, RANGE_MILES, PRESSURIZED)
AIRLINE(AIRLINE_CODE, AIRLINE_NAME, NOTE)
AIRPORT(AIRPORT_CODE, AIRPORT_NAME, AIRPORT_LOCATION, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE, MINIMUM_CONNECT_TIME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE, MILES_DISTANT, DIRECTION, MINUTES_DISTANT)
CITY(CITY_CODE, CITY_NAME, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE)
CLASS_OF_SERVICE(BOOKING_CLASS, RANK, CLASS_DESCRIPTION)
CODE_DESCRIPTION(CODE, DESCRIPTION)
COMPARTMENT_CLASS(COMPARTMENT, CLASS_TYPE)
DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)
DAYS(DAYS_CODE, DAY_NAME)
DUAL_CARRIER(MAIN_AIRLINE, LOW_FLIGHT_NUMBER, HIGH_FLIGHT_NUMBER, DUAL_AIRLINE, SERVICE_N

 33%|███▎      | 1/3 [00:02<00:05,  2.97s/it]

🧪 Q: i need a flight from DENVER to SALT LAKE CITY on monday
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, CITY AS CITYalias0, CITY AS CITYalias1, DATE_DAY AS DATE_DAYalias0, DAYS AS DAYSalias0, FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""SALT LAKE CITY"" AND DATE_DAYalias0.DAY_NUMBER = 7 AND DATE_DAYalias0.MONTH_NUMBER = 1 AND DATE_DAYalias0.YEAR = 1991 AND DAYSalias0.DAY_NAME = DATE_DAYalias0.DAY_NAME AND FLIGHTalias0.FLIGHT_DAYS = DAYSalias0.DAYS_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""DENVER"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , C

 67%|██████▋   | 2/3 [00:05<00:02,  2.97s/it]

🧪 Q: list flights from MILWAUKEE to SAN JOSE on wednesday
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , DAYS AS DAYSalias0 , DAYSalias1 , CITY AS CITYalias2 , CITY AS CITYalias3 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""MILWAUKEE"" AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""SAN JOSE"" AND DAYSalias0.DAY_NAME = ""WEDNESDAY"" AND DAYSalias1.DAY_NAME = ""WEDNESDAY"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , DATE_DAY AS DATE_DAYalias0 , DAYS AS DAYSalias0 , FLIGHT

100%|██████████| 3/3 [00:15<00:00,  5.04s/it]

🧪 Q: list flights from SAN JOSE to DALLAS on friday
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , DATE_DAY AS DATE_DAYalias0 , DAYS AS DAYSalias0 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = "SAN JOSE" AND DATE_DAYalias0.DAY_NUMBER = 5 AND DATE_DAYalias0.MONTH_NUMBER = 4 AND DATE_DAYalias0.YEAR = 2023 AND DAYSalias0.DAY_NAME = DATE_DAYalias0.DAY_NAME AND FLIGHTalias0.FLIGHT_DAYS = DAYSalias0.DAYS_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = "DALLAS" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYal